# 문과 고민 크롤링

In [ ]:
import requests
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
from Data_Analysis import *


In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/mentoring?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_mentoring_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [100]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/mentoring/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_mentoring_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_QA.json', orient='records', force_ascii=False, indent=4)


Scraping posts:  33%|███▎      | 1/3 [00:06<00:12,  6.49s/post]

Post ID: 4423268, Title: 인턴 서류평가시 가장 중요하게 평가되는 항목, Best Rate: 56.0, Best Comment: 안녕하세요. 인턴 서류에서 가장 중요하게 평가되는 항목은 지원하는 회사와 직무에 따라 조금...


Scraping posts:  67%|██████▋   | 2/3 [00:12<00:06,  6.11s/post]

Post ID: 4423041, Title: 영업교육 경력을 기재하는게 좋을지 빼는게 좋을지 고민입니다., Best Rate: 55.0, Best Comment: 저라면 쓸거같아요. 왜냐하면 영업관리직무 지원하면 점포,매장관리 및 교육을 기본적으로 하거...


Scraping posts: 100%|██████████| 3/3 [00:17<00:00,  5.98s/post]

Post ID: 4422416, Title: 면접에서 지원한 직무랑 안어울린다는 말을 들었어요 .., Best Rate: 56.0, Best Comment: 안녕하세요. 면접관도 사람인지라, 짧은 시간 안에 지원자를 완벽하게 파악하기는 어렵습니다....


# 이과 고민 크롤링

In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/STEM_mentoring?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_STEM_mentoring_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [ ]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/STEM_mentoring/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_STEM_mentoring_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_STEM_QA.json', orient='records', force_ascii=False, indent=4)


# 대학생 고민

In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/mento_campus?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_mento_campus_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [ ]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/mento_campus/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_mento_campus_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_campus_QA.json', orient='records', force_ascii=False, indent=4)


# 직장인 고민

In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/mentor_employed?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_mento_employee_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [ ]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/mentor_employed/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_mento_employee_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_employee_QA.json', orient='records', force_ascii=False, indent=4)


# 삼성 질문

In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/mento_samsung?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_samsung_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [ ]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/mento_samsung/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_samsung_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_samsung_QA.json', orient='records', force_ascii=False, indent=4)


# SK 질문

In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/mento_sk?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_sk_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [ ]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/mento_sk/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_sk_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_sk_QA.json', orient='records', force_ascii=False, indent=4)


# 현대

In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/mento_hyndai?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_hyundai_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [ ]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/mento_hyndai/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_hyundai_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_hyundai_QA.json', orient='records', force_ascii=False, indent=4)


# LG 질문

In [ ]:
page = 1
data_row_keys_css = []

with tqdm(desc="페이지 스크래핑 중", unit="page") as pbar:
    while True:
        url = f"https://community.linkareer.com/mento_lg?page={page}"
        html = requests.get(url)
        soup = BeautifulSoup(html.content, "html.parser")

        # 해당 페이지에서 필터링된 키를 임시로 저장할 리스트
        current_page_filtered_keys = []

        # rc-table-row 와 rc-table-row-level-0 클래스를 모두 가진 모든 tr 태그를 선택
        tr_tags_with_base_classes = soup.select('tr.rc-table-row.rc-table-row-level-0')

        for tr in tr_tags_with_base_classes:
            # 'rc-notice' 클래스가 없고, data-row-key 속성이 있는 경우에만 추출
            if 'rc-notice' not in tr.get('class', []) and 'data-row-key' in tr.attrs:
                key = tr['data-row-key']
                current_page_filtered_keys.append(key)
                data_row_keys_css.append(key)  # 최종 결과 리스트에 추가

        # 현재 페이지에서 추출된 필터링된 키가 하나도 없으면 루프 종료
        if not current_page_filtered_keys:
            break

        pbar.set_postfix(current_page=page, total_keys=len(data_row_keys_css))
        pbar.update(1)  # 한 페이지가 처리될 때마다 진행바 업데이트
        page += 1

print(f"총 스크래핑된 페이지: {page - 1}")
print(f"총 data_row_keys: {len(data_row_keys_css)}")

In [ ]:
df = pd.DataFrame(data_row_keys_css)

df.to_json('linkareer_lg_data_row_keys.json', orient='records', force_ascii=False, indent=4)

In [ ]:
import json
import re
import time
import pandas as pd
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


def scrape_post(post_id: int, driver: webdriver.Chrome) -> dict:
    url = f"https://community.linkareer.com/mento_lg/{post_id}?page=1"
    driver.get(url)
    time.sleep(5)

    # 1) 정확한 제목 셀렉터
    try:
        title = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-container .header-title"
        ).text.strip()
    except:
        title = ""

    # 2) 정확한 본문 셀렉터
    try:
        content = driver.find_element(
            By.CSS_SELECTOR,
            "#post-detail-content-container .post-detail"
        ).text.strip()
    except:
        content = ""

    # 3) 댓글 셀렉터도 확인 후 업데이트
    best_rate = -1.0
    best_comment = ""
    for li in driver.find_elements(By.CSS_SELECTOR, "#post-detail-comments-section ul li"):
        try:
            # (이 부분도 실제 클래스명에 맞게 변경)
            rate_txt = li.find_element(
                By.CSS_SELECTOR,
                ".sc-7450030d-0.geJdfg p"
            ).text
            rate = float(re.search(r'(\d+)', rate_txt).group(1))
            if rate > best_rate:
                best_rate = rate
                best_comment = li.find_element(
                    By.CSS_SELECTOR,
                    ".ProseMirror.content.cmt-content.sc-f213cdb-1.jHRyml"
                ).text.strip()
        except:
            continue
    print(f"Post ID: {post_id}, Title: {title}, Best Rate: {best_rate}, Best Comment: {best_comment[:50]}...")

    return {
        "post_id": post_id,
        "title": title,
        "content": content,
        "best_rate": best_rate,
        "best_comment": best_comment
    }


def main(post_ids: list[int]) -> pd.DataFrame:
    chrome_opts = Options()
    chrome_opts.add_argument("--headless")
    chrome_opts.add_argument("--no-sandbox")
    chrome_opts.add_argument("--disable-dev-shm-usage")
    chrome_opts.add_argument("--window-size=1920,1080")

    service = Service(executable_path="/opt/homebrew/bin/chromedriver")
    driver = webdriver.Chrome(service=service, options=chrome_opts)

    rows = []
    try:
        for pid in tqdm(post_ids, desc="Scraping posts", unit="post"):
            rows.append(scrape_post(pid, driver))
    finally:
        driver.quit()

    return pd.DataFrame(rows, columns=["post_id", "title", "content", "best_rate", "best_comment"])


if __name__ == "__main__":
    with open('linkareer_lg_data_row_keys.json', 'r', encoding='utf-8') as f:
        raw = json.load(f)

    post_ids = [int(d['0']) for d in raw]

    # post_ids각 요소들의 뒷 두자리는 삭제
    real_ids = [int(str(pid)[:-2]) for pid in post_ids]

    real_ids

    df = main(real_ids[:3])
    df.to_json('linkareer_lg_QA.json', orient='records', force_ascii=False, indent=4)
